# Banking Support Chatbot - End to End Machine Learning Workshop

**Workshop Days 2, 3 and 4 | Complete Notebook: Steps 1 to 20**

---

### What We Are Building
A Banking Support Chatbot that takes a customer question in plain English and returns the correct bank response automatically using Machine Learning, deployed as a live shareable web app.

### Dataset
File: banking_support.csv
20 rows of real banking support tickets. Columns: id, ticket_id, category, priority, question, response

### The Full 20-Step Roadmap

| Day | Steps | Theme |
|-----|-------|-------|
| Day 2 | 1 to 8 | Data loading, EDA, text preprocessing, TF-IDF |
| Day 3 | 9 to 15 | Train-test split, Naive Bayes, predictions, evaluation, response function, model comparison |
| Day 4 | 16 to 20 | Retrain with more data, save model, Gradio chatbot UI, stress test, course roadmap |

This single notebook runs Steps 1 through 20 end to end, top to bottom, in Google Colab. Steps 1 to 8 below are exactly what was covered on Day 2. Step 9 onward continues straight into Day 3 and Day 4 in the same notebook so the whole build can be demonstrated live in one session.

---

## STEP 1 - Big Picture Walkthrough

Before writing any code, the instructor will walk through all 20 steps of this project.

| Day | Steps | Theme |
|-----|-------|-------|
| Day 2 Today | 1 to 8 | Data loading, EDA, text preprocessing, TF-IDF |
| Day 3 | 9 to 15 | Model training, predictions, evaluation, response function |
| Day 4 | 16 to 20 | Retrain, save model, Gradio chatbot UI, deploy |

The final output is a live chatbot web app with a shareable link that answers banking queries automatically.

## STEP 2 - Google Colab Setup

You are already inside Google Colab. A few things to know:
- Each grey box is a code cell. Press Shift+Enter to run it.
- The number in brackets on the left shows the order cells were run.
- If there is an error, read the last line of the red message first.
- Always run cells from top to bottom. Never skip a cell.

Run the cell below to confirm Python is working.

In [ ]:
# Confirm Python is working
print('Setup complete. Python is running.')
print('We are building a Banking Support Chatbot using Machine Learning.')

## STEP 3 - Upload Dataset and Read with Pandas

How to upload the file:
1. Click the folder icon on the left sidebar in Colab
2. Click the upload icon (paper with arrow pointing up)
3. Select banking_support.csv from your computer
4. Wait for the upload to finish
5. The file will appear in the file panel on the left

Then run the cells below.

In [ ]:
# Import the Pandas library
# Pandas lets us work with tables of data in Python
import pandas as pd

print('Pandas imported successfully.')

In [ ]:
# Read the CSV file into a DataFrame
# A DataFrame is like an Excel table: rows and columns
df = pd.read_csv('banking_support.csv')

# Show the first 5 rows
df.head()

In [ ]:
# Show all 20 rows to get a full view of the dataset
df

## STEP 4 - Understand the Data Structure

Before building any model, we must understand exactly what data we have.
This step answers: How many rows? How many columns? What type of data is in each column?

In [ ]:
# Shape returns (number of rows, number of columns)
print('Shape of the dataset:')
print(df.shape)
print()
print('We have', df.shape[0], 'rows and', df.shape[1], 'columns.')

In [ ]:
# Column names
print('Column names:')
print(df.columns.tolist())

In [ ]:
# Data types of each column
# object means text string, int64 means whole number
print('Data types:')
print(df.dtypes)

In [ ]:
# Detailed info about the DataFrame
print('Detailed information:')
df.info()

In [ ]:
# Check if any values are missing
# Missing values would cause errors during model training
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Zero missing values means the dataset is complete and clean.')

In [ ]:
# Look at one sample question and response to understand the content
print('Sample Question:')
print(df['question'][0])
print()
print('Sample Response:')
print(df['response'][0])
print()
print('Category:', df['category'][0])
print('Priority:', df['priority'][0])

### Key Observations from the Data

- The **question** column is the input to our ML model
- The **category** column is the label our model will predict (the intent)
- The **response** column is the answer we will return to the user after prediction
- There are 20 unique categories, one per row in this dataset
- All data is text which means we must convert it to numbers before feeding to ML

## STEP 5 - EDA Part 1: Category Distribution

EDA stands for Exploratory Data Analysis.
Before building any model, we visualize the data to understand it.

Question we are answering: What types of banking problems do customers ask about?

In [ ]:
# Import the visualization library
import matplotlib.pyplot as plt

print('Matplotlib imported successfully.')

In [ ]:
# Count how many rows belong to each category
print('Category distribution:')
print(df['category'].value_counts())
print()
print('Total unique categories:', df['category'].nunique())

In [ ]:
# Plot a bar chart of category distribution
plt.figure(figsize=(14, 5))
df['category'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Number of Tickets per Category', fontsize=14)
plt.xlabel('Category (Intent)')
plt.ylabel('Number of Tickets')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Each category has exactly 1 ticket.')
print('The model must learn to distinguish all 20 categories from just the question text.')

In [ ]:
# List all categories clearly with numbers
print('All 20 intent categories the chatbot will handle:')
print()
for i, cat in enumerate(df['category'].tolist(), 1):
    print(f'{i:>2}. {cat}')

## STEP 6 - EDA Part 2: Priority Distribution and Question Length

We now look at two more things:
1. How urgent are these queries? (priority column)
2. How long are the customer questions? (length of question text)

In [ ]:
# Priority distribution
print('Priority distribution:')
print(df['priority'].value_counts())
print()

# Which categories are high priority?
print('High priority categories:')
print(df[df['priority'] == 'high']['category'].tolist())

In [ ]:
# Plot priority as a pie chart
priority_counts = df['priority'].value_counts()
colors = ['#e74c3c', '#f39c12', '#2ecc71']

plt.figure(figsize=(6, 6))
plt.pie(priority_counts, labels=priority_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90)
plt.title('Ticket Priority Distribution', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compute the character length of each question
df['question_length'] = df['question'].apply(len)

print('Question length statistics:')
print(df['question_length'].describe().round(1))
print()
print('Shortest question:')
print(df.loc[df['question_length'].idxmin(), 'question'])
print()
print('Longest question:')
print(df.loc[df['question_length'].idxmax(), 'question'])

In [ ]:
# Plot question length distribution as a histogram
plt.figure(figsize=(10, 4))
plt.hist(df['question_length'], bins=10, color='steelblue', edgecolor='white')
plt.title('Distribution of Question Lengths (number of characters)', fontsize=14)
plt.xlabel('Number of Characters')
plt.ylabel('Number of Questions')
plt.tight_layout()
plt.show()

In [ ]:
# Compare average question length by priority level
print('Average question length by priority:')
print(df.groupby('priority')['question_length'].mean().round(1))

## STEP 7 - Text Preprocessing

**Why do we need to preprocess text?**

Raw text has noise that does not help the ML model:
- Capital letters: Account and account should be treated as the same word
- Punctuation: question marks and commas carry no intent meaning
- Stopwords: words like my, the, is, what, should appear in every sentence and carry no useful information

We will clean the question column in three steps:
1. Convert to lowercase
2. Remove punctuation using regular expressions
3. Remove stopwords using NLTK

In [ ]:
# Import NLTK - Natural Language Toolkit
# This is the standard Python library for text processing
import nltk
import re

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

from nltk.corpus import stopwords

print('NLTK imported successfully.')

In [ ]:
# See what English stopwords look like
english_stopwords = set(stopwords.words('english'))
print('Total English stopwords:', len(english_stopwords))
print()
print('Sample stopwords:')
print(sorted(list(english_stopwords))[:30])

In [ ]:
# Define the text cleaning function
# This function takes one question string and returns a cleaned version

def clean_text(text):
    # Step 1: Convert to lowercase
    text = text.lower()
    
    # Step 2: Remove punctuation and special characters, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 3: Remove stopwords
    words = text.split()
    words = [w for w in words if w not in english_stopwords]
    
    # Step 4: Join the remaining words back into a sentence
    cleaned = ' '.join(words)
    
    return cleaned

print('clean_text function defined.')

In [ ]:
# Test the function on one example before applying to the full dataset
raw_question = 'My account is locked what should I do?'
cleaned_question = clean_text(raw_question)

print('BEFORE cleaning:')
print(raw_question)
print()
print('AFTER cleaning:')
print(cleaned_question)
print()
print('Words removed: my, is, what, should, i, do')
print('Words kept: account, locked')
print('These two words are enough for the model to identify the intent.')

In [ ]:
# Apply the cleaning function to all 20 questions
# This creates a new column called clean_question
df['clean_question'] = df['question'].apply(clean_text)

print('Cleaning applied to all rows.')
print()
print('Before and after for first 5 rows:')
print()
for i in range(5):
    print(f'Row {i+1}:')
    print(f'  BEFORE: {df["question"][i]}')
    print(f'  AFTER:  {df["clean_question"][i]}')
    print()

In [ ]:
# View the updated DataFrame with the new clean_question column
df[['question', 'clean_question', 'category']].head(10)

## STEP 8 - TF-IDF Vectorization: Converting Text to Numbers

**The core problem:** Machine Learning models work with numbers, not words.

**The solution:** TF-IDF converts each question into a row of numbers.

**How TF-IDF works:**
- Every unique word in the dataset becomes one column
- Every question becomes one row
- The number at each cell = how important that word is for that question
- Common words get low scores. Rare specific words get high scores.

TF = Term Frequency: how often this word appears in this question
IDF = Inverse Document Frequency: how rare this word is across all questions
TF-IDF score = TF multiplied by IDF

In [ ]:
# Import TfidfVectorizer from scikit-learn
# scikit-learn is the main Machine Learning library in Python
from sklearn.feature_extraction.text import TfidfVectorizer

print('scikit-learn TfidfVectorizer imported successfully.')

In [ ]:
# Create the TF-IDF vectorizer
# max_features=500 means we keep only the 500 most informative words
vectorizer = TfidfVectorizer(max_features=500)

# fit_transform does two things:
# fit = learns the vocabulary from our questions
# transform = converts each question into a row of numbers
X = vectorizer.fit_transform(df['clean_question'])

print('TF-IDF vectorization complete.')
print()
print('Shape of X (feature matrix):', X.shape)
print()
print(f'{X.shape[0]} rows = one row per question')
print(f'{X.shape[1]} columns = one column per unique word in the dataset')

In [ ]:
# Define y - the target labels that the model will learn to predict
# y is the category column from the original DataFrame
y = df['category']

print('Target variable y defined.')
print()
print('y contains the category for each of the 20 questions:')
print(y.tolist())

In [ ]:
# See which words the vectorizer learned from our dataset
feature_names = vectorizer.get_feature_names_out()
print('Total words in vocabulary:', len(feature_names))
print()
print('First 30 words in alphabetical order:')
print(sorted(feature_names)[:30])

In [ ]:
# Convert the matrix to a readable DataFrame to understand it
import numpy as np
X_df = pd.DataFrame(X.toarray(), columns=feature_names)

# Show a small portion - rows are questions, columns are words, values are TF-IDF scores
print('TF-IDF matrix - first 5 rows, first 10 columns:')
print('Rows = questions, Columns = words, Values = importance scores')
X_df.iloc[:5, :10]

In [ ]:
# Show top scoring words for the first question
first_question_scores = X_df.iloc[0]
top_words = first_question_scores[first_question_scores > 0].sort_values(ascending=False)

print('Original question:', df['question'][0])
print('Cleaned question:', df['clean_question'][0])
print()
print('Top TF-IDF words and scores for this question:')
print(top_words.head(10))

In [ ]:
# Test the vectorizer on a completely new unseen question
# This simulates what happens when the chatbot receives a user query

new_question = 'my debit card is not working at the ATM'
new_question_cleaned = clean_text(new_question)
new_question_vectorized = vectorizer.transform([new_question_cleaned])

print('New question (raw):', new_question)
print('New question (cleaned):', new_question_cleaned)
print('Vectorized shape:', new_question_vectorized.shape)
print()
print('The question is now a row of numbers.')
print('Tomorrow we will feed this into the ML model to predict the category.')

---

## Day 2 Complete - Summary

| Step | What We Did | Result |
|------|-------------|--------|
| Step 3 | Loaded banking_support.csv | DataFrame with 20 rows and 6 columns |
| Step 4 | Explored the data structure | Understood question input, category label, response output |
| Step 5 | EDA - category distribution | 20 categories, all intent types identified and visualized |
| Step 6 | EDA - priority and question length | 8 high priority queries, question lengths 30 to 60 characters |
| Step 7 | Text preprocessing | Cleaned questions: lowercase, no punctuation, no stopwords |
| Step 8 | TF-IDF vectorization | Feature matrix X and target array y ready for ML |

### Variables ready for Day 3

- X - the TF-IDF feature matrix, input to the ML model
- y - the category labels, what the model will learn to predict
- vectorizer - the fitted TF-IDF vectorizer, needed to transform new questions
- df - the full DataFrame, needed to retrieve responses after prediction

### Day 3 will cover Steps 9 to 15
Train-test split, Naive Bayes model training, predictions, evaluation, response retrieval function, unknown query handling, model comparison

---

Save this notebook now: File > Save a Copy in Drive
Rename it to banking_chatbot_yourname.ipynb

---

## Continuing into Day 3 and Day 4 - Steps 9 to 20

Day 2 ended with a clean TF-IDF feature matrix `X` and target labels `y`, built from the DataFrame `df`. We pick up exactly where we left off. Do not re-upload the CSV or re-run Steps 1 to 8 unless your Colab runtime was disconnected - if `df`, `X`, `y`, `vectorizer` and `clean_text` are still in memory, just continue running cells from here.

If you did restart your runtime, simply run all the Step 1 to 8 cells above first (Runtime menu -> Run before), then continue below.

## STEP 9 - Train-Test Split

**The textbook and exam analogy:** Imagine a student who studies from a textbook and is then tested with the exact same questions from the textbook. Scoring 100% tells you nothing about whether the student actually understood the subject - they may have just memorized the answers. A fair test uses questions the student has never seen before.

Machine learning works the same way. If we train a model and test it on the same data, we cannot tell if it has truly learned the underlying patterns or has simply memorized the training examples. This is called **overfitting**: the model performs great on data it has already seen, but poorly on new, unseen data.

To get an honest estimate of how well our model will perform on new customer questions, we split our dataset into two parts:
- **Training set (80%)** - the textbook the model studies from
- **Test set (20%)** - the unseen exam questions used only to check performance

**An important honesty note for this dataset:** We only have 20 rows, and every single row belongs to a *different* category (20 unique categories). This is a very small dataset for a 20-class problem. When we randomly hold out 20% as a test set, some categories may end up *only* in the test set and never appear in the training set at all. If that happens, the model has literally never seen that category during training and cannot possibly predict it correctly. Keep this in mind - it becomes an important and very real lesson in Step 12.

In [ ]:
# Import the train-test split function from scikit-learn
from sklearn.model_selection import train_test_split

# Split X (features) and y (labels) into training and test sets
# test_size=0.2 means 20% of rows go into the test set
# random_state=42 makes the split reproducible - everyone gets the same split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)
print()
print(f'{X_train.shape[0]} questions to train on, {X_test.shape[0]} questions to test on.')

In [ ]:
# Let's see exactly which categories ended up in training vs testing
# This uses the original index positions to look back at df['category']
train_categories = sorted(df.loc[y_train.index, 'category'].tolist())
test_categories = sorted(df.loc[y_test.index, 'category'].tolist())

print('Categories in the TRAINING set (the model will learn these):')
print(train_categories)
print()
print('Categories in the TEST set (used only to check performance):')
print(test_categories)
print()
print('Notice: the categories in the test set do NOT appear in the training list above.')
print('This is the small-dataset challenge we just discussed - keep this in mind for Step 12.')

## STEP 10 - Train Naive Bayes Model

**What is Naive Bayes?** It is a simple, fast, and surprisingly effective algorithm for text classification. It is based on probability: it looks at every word in a question and calculates how likely each category is, given those words. For example, if the word "locked" mostly appeared in "account" category questions during training, then any new question containing "locked" gets a higher probability score for the "account" category.

It is called "naive" because it makes a simplifying assumption: it treats every word as independent of every other word (it does not consider word order or grammar). This assumption is technically wrong, but in practice it works very well for short texts like customer support questions.

**What does "fitting" a model mean?** Fitting (also called training) means showing the model the training questions along with their correct categories, and letting it calculate the word-to-category probabilities. Once fitted, the model can use those probabilities to predict the category of a brand new question it has never seen.

In [ ]:
# Import the Naive Bayes classifier for text data
from sklearn.naive_bayes import MultinomialNB

# Create the model
nb_model = MultinomialNB()

# Fit (train) the model on the training data only
# The model learns which words are associated with which categories
nb_model.fit(X_train, y_train)

print('Naive Bayes model trained successfully.')
print('The model has learned word-to-category patterns from', X_train.shape[0], 'training questions.')
print()
print('Categories the model knows about (learned from the training set):')
print(sorted(nb_model.classes_.tolist()))

## STEP 11 - Make Predictions

Now that the model is trained, let's use it to predict categories. First we run it on the test set (the unseen exam questions) and compare predictions to the actual answers. Then, for the live hands-on moment, we type a brand new question that was never in our dataset at all and watch the model classify it in real time.

In [ ]:
# Predict categories for every question in the test set
test_predictions = nb_model.predict(X_test)

# Put actual vs predicted side by side so we can compare easily
comparison = pd.DataFrame({
    'Actual Category': y_test.values,
    'Predicted Category': test_predictions
})
comparison

In [ ]:
# STUDENT HANDS-ON MOMENT
# Try changing the sentence below to any banking question and re-run this cell.
# Watch the model predict a category for a sentence it has never seen before.

live_question = 'My debit card was declined while shopping at a store'

# Run it through the exact same pipeline we built on Day 2:
# clean the text -> convert to TF-IDF numbers -> predict
live_question_cleaned = clean_text(live_question)
live_question_vectorized = vectorizer.transform([live_question_cleaned])
live_prediction = nb_model.predict(live_question_vectorized)[0]

# predict_proba gives the model's confidence for every category it knows
live_probabilities = nb_model.predict_proba(live_question_vectorized)[0]
top_3 = sorted(zip(nb_model.classes_, live_probabilities), key=lambda pair: -pair[1])[:3]

print('Live question:', live_question)
print('Cleaned version:', live_question_cleaned)
print()
print('Predicted category:', live_prediction)
print()
print('Top 3 most likely categories with confidence scores:')
for category, score in top_3:
    print(f'  {category:12s} {score*100:.1f}%')

## STEP 12 - Evaluate Model

Now we measure exactly how good our model is, using three standard tools:
- **Accuracy** - what percentage of test questions did the model classify correctly overall
- **Precision** - of everything the model labeled as category X, what percentage was actually category X
- **Recall** - of everything that was actually category X, what percentage did the model correctly find
- **Confusion matrix** - a grid showing exactly which categories get confused with which other categories

**Important reality check before we run this:** As flagged in Step 9, our test set contains categories the model never saw during training (look back at the printed lists - the test categories were not in the training categories). A model literally cannot predict a label it has never been shown, no matter how good the algorithm is. So do not be alarmed if the accuracy below looks very low - this is a direct, honest, real-world consequence of having only 20 rows for a 20-class problem, not a mistake in our code. It is, in fact, one of the most important lessons in this entire workshop: **a machine learning model is only as good as the data it is trained on, and it needs multiple examples per category to generalize.** We will act on this lesson directly in Step 16.

In [ ]:
# Import evaluation tools from scikit-learn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Accuracy: percentage of correct predictions on the test set
test_accuracy = accuracy_score(y_test, test_predictions)
print(f'Test set accuracy: {test_accuracy*100:.1f}%')
print()
print('Why so low? Every single category in our test set was completely absent from the')
print('training set (see Step 9). The model cannot predict a category it has never seen,')
print('no matter how well it is trained. This is a data quantity problem, not a model problem.')

In [ ]:
# Precision, recall, and f1-score for every category
# zero_division=0 avoids warning messages for categories with no predictions
print('Detailed classification report:')
print(classification_report(y_test, test_predictions, zero_division=0))

In [ ]:
# Visualize the confusion matrix as a heatmap
# Rows = actual category, Columns = predicted category
# A perfect model would show all values on the diagonal

all_labels = sorted(set(y_test.tolist()) | set(test_predictions.tolist()))
cm = confusion_matrix(y_test, test_predictions, labels=all_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=all_labels, yticklabels=all_labels)
plt.title('Confusion Matrix - Test Set Predictions', fontsize=13)
plt.xlabel('Predicted Category')
plt.ylabel('Actual Category')
plt.tight_layout()
plt.show()

print('No values appear on the diagonal, confirming the model could not match any test')
print('category, since none of them were present in the training data.')

In [ ]:
# SANITY CHECK: does the model actually learn anything at all, when given the chance?
# We train a fresh model on ALL 20 rows (no holdout) and check it against the SAME data
# This is training accuracy, not test accuracy - it does NOT measure generalization.
# It only proves the model is capable of learning the patterns when it has seen them.

model_full = MultinomialNB()
model_full.fit(X, y)

training_predictions = model_full.predict(X)
training_accuracy = accuracy_score(y, training_predictions)

print(f'Training accuracy (model evaluated on data it was trained on): {training_accuracy*100:.1f}%')
print()
print('This confirms the algorithm itself works correctly and CAN separate all 20 categories')
print('when it has seen at least one example of each. The earlier low test accuracy was')
print('purely a consequence of unseen categories in the test split, not a flaw in Naive Bayes.')

### Key Takeaway from Step 12

- The train/test split honestly exposed a real limitation: **20 rows for 20 categories means roughly 1 example per category, which is not enough for a model to generalize to brand-new test categories.**
- The sanity check confirms the model architecture itself works perfectly well once it has seen an example of a category.
- **From this point forward**, we will use `model_full` (trained on all 20 rows) to power our actual chatbot, because in a real deployment we want our assistant to recognize every category we have data for, not withhold 20% of our already-tiny dataset from it. Train/test splitting was the right tool to *honestly evaluate* our pipeline; using all available data is the right choice for the *deployed* chatbot. This is standard real-world practice: validate with a split, then train the final model on everything before shipping it.

## STEP 13 - Build Response Retrieval Function

This is the heart of the chatbot. The logic is simple and is the same idea behind every FAQ-bot and support-bot you have ever used:

1. Take the customer's raw question
2. Clean it the same way we cleaned our training data
3. Convert it to TF-IDF numbers using our already-fitted `vectorizer`
4. Use `model_full` to predict the category
5. Look up the official bank response for that category in our DataFrame
6. Return the response

We will use `model_full` (trained on all 20 rows in Step 12) since this is the version we are deploying.

In [ ]:
def get_response_basic(question, model, vectorizer, df):
    """
    Takes a raw customer question and returns the bank's official response.
    Steps: clean -> vectorize -> predict category -> look up response.
    """
    cleaned = clean_text(question)
    vector = vectorizer.transform([cleaned])
    predicted_category = model.predict(vector)[0]

    # Look up the response for the predicted category in our DataFrame
    response = df[df['category'] == predicted_category]['response'].iloc[0]

    return response, predicted_category

print('get_response_basic() function defined. The core chatbot logic is complete.')

In [ ]:
# Try it out on a few questions
sample_questions = [
    'My account is locked, what should I do?',
    'How can I transfer money to my friend?',
    'I lost my debit card',
]

for q in sample_questions:
    response, category = get_response_basic(q, model_full, vectorizer, df)
    print('Question:', q)
    print('Predicted category:', category)
    print('Chatbot response:', response)
    print()

## STEP 14 - Handle Unknown Queries

What happens if a customer asks something completely unrelated to banking, like "what's the weather today"? Right now, `get_response_basic` would still confidently return *some* category and *some* response, because `model.predict()` always picks the single best-matching category from the 20 it knows, even if none of them are a good match. That is a real-world edge case we must handle, otherwise our chatbot will confidently give wrong answers to questions it has no business answering.

**The fix: confidence thresholds.** Instead of just taking the top prediction, we look at `predict_proba()`, which gives a probability score for every one of the 20 categories. If the model's best score is still very low, that tells us it could not find a good match, and we should show a fallback "please contact our helpline" message instead of guessing.

**How do we pick the right threshold?** Let's actually look at real confidence scores first.

In [ ]:
# Let's compare confidence scores for clearly in-scope vs clearly out-of-scope questions
# to figure out a sensible threshold

probe_questions = [
    ('My account is locked, what should I do?', 'in-scope'),
    ('Why is my credit card payment not showing?', 'in-scope'),
    ('What is the weather like today?', 'out-of-scope'),
    ('Can you book me a flight ticket?', 'out-of-scope'),
    ('Tell me a joke', 'out-of-scope'),
]

for q, kind in probe_questions:
    cleaned = clean_text(q)
    vec = vectorizer.transform([cleaned])
    proba = model_full.predict_proba(vec)[0]
    top_score = proba.max()
    print(f'[{kind:12s}] top confidence = {top_score*100:4.1f}%   question: {q}')

print()
print('Notice: in-scope questions score noticeably above the 1/20 = 5.0% baseline,')
print('while completely out-of-scope questions sit right at exactly 5.0%.')
print('Why exactly 5.0%? With 20 equally-likely categories and zero recognizable words,')
print('the model has no information to prefer any one category, so it falls back to the')
print('uniform baseline of 1 divided by 20 categories. We will use this fact directly.')

In [ ]:
# Final response function with a confidence threshold and a fallback for unknown queries

CONFIDENCE_THRESHOLD = 0.06  # 6% - just above the 5.0% no-information baseline we found above
FALLBACK_MESSAGE = ("I'm sorry, I could not confidently understand your question. "
                     "Please call our 24x7 customer helpline at 1800-419-1234 for assistance.")

def get_response(question, model, vectorizer, df, threshold=CONFIDENCE_THRESHOLD):
    """
    Production-ready version of our chatbot logic.
    Returns (response_text, predicted_category, confidence_score).
    predicted_category will be None if the question could not be matched at all.
    """
    cleaned = clean_text(question)
    vector = vectorizer.transform([cleaned])

    # Safety check 1: the question shares ZERO words with anything in our vocabulary
    if vector.sum() == 0:
        return FALLBACK_MESSAGE, None, 0.0

    # Safety check 2: the model recognized some words, but its best guess is still weak
    probabilities = model.predict_proba(vector)[0]
    best_index = probabilities.argmax()
    confidence = probabilities[best_index]
    predicted_category = model.classes_[best_index]

    if confidence < threshold:
        return FALLBACK_MESSAGE, predicted_category, confidence

    response = df[df['category'] == predicted_category]['response'].iloc[0]
    return response, predicted_category, confidence

print('get_response() defined with confidence-based fallback handling.')

In [ ]:
# Test the upgraded function on both in-scope and out-of-scope questions
final_test_questions = [
    'My account is locked, what should I do?',
    'I lost my debit card',
    'What is the weather like today?',
    'Tell me a joke',
]

for q in final_test_questions:
    response, category, confidence = get_response(q, model_full, vectorizer, df)
    print('Question:', q)
    print(f'Category: {category}   Confidence: {confidence*100:.1f}%')
    print('Response:', response)
    print()

## STEP 15 - Compare Other Models

Naive Bayes is not the only algorithm that can classify text. Two other very popular choices are:

- **Logistic Regression** - despite the name, this is a classification algorithm. It draws a mathematical boundary between categories based on the TF-IDF feature weights.
- **Linear SVC (Support Vector Machine)** - tries to find the boundary line that separates categories with the widest possible margin, which often makes it robust on text data.

**The lesson here is not "which model wins" - it is that there is no single best model for every problem.** Different algorithms make different assumptions, and the only way to know which works best for your specific data is to experiment and compare. We will train all three on the exact same train/test split for a fair comparison.

In [ ]:
# Import the two new models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Train all three models on the SAME training data for a fair comparison
models_to_compare = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Linear SVM': LinearSVC(max_iter=2000),
}

results = {}
for name, model in models_to_compare.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    results[name] = acc

print('Model comparison on the same test set:')
print()
for name, acc in results.items():
    print(f'{name:22s} accuracy = {acc*100:.1f}%')

### Key Takeaway from Step 15

All three models score the same on this particular test set. That is expected and instructive: **no algorithm can correctly predict a category it has never seen during training.** Switching algorithms cannot fix a fundamental shortage of training examples per category - only more data can. This sets up Step 16 perfectly: let's add more data and see what actually moves the needle.

## STEP 16 - Add More Data and Retrain

This is the **ML feedback loop**: collect data, train, evaluate, find weaknesses, collect more data targeted at those weaknesses, retrain, and check if things improved. Real production systems repeat this loop continuously as they receive more real customer conversations.

We will manually add 5 new question-and-response rows for 5 of our existing categories (`account`, `transfer`, `balance`, `card`, `upi`), written in different wording than the original rows. This gives those 5 categories **2 training examples instead of 1**, while the other 15 categories still have only 1.

Then we will test both the **old model** (trained on the original 20 rows) and a **new model** (trained on all 25 rows) against 5 brand-new questions - written in yet a third, different style, never seen by either model - to see whether the additional data actually helps the model generalize.

In [ ]:
# Define the 5 new rows as a list of dictionaries
new_rows = [
    {'id': 21, 'ticket_id': 'BK021', 'category': 'account', 'priority': 'high',
     'question': 'My net banking login is locked, how do I unlock my account?',
     'response': 'Unlock instantly using OTP verification at Forgot Password, or visit your nearest branch with valid ID proof if the issue continues.'},
    {'id': 22, 'ticket_id': 'BK022', 'category': 'transfer', 'priority': 'medium',
     'question': 'Please send five thousand rupees to my brother using NEFT.',
     'response': 'Open Mobile App -> Transfer Money -> Enter beneficiary account/IFSC -> Amount -> Confirm with OTP. NEFT transfers complete within 30 minutes.'},
    {'id': 23, 'ticket_id': 'BK023', 'category': 'balance', 'priority': 'low',
     'question': 'Can you show me my savings account balance instantly?',
     'response': 'Login to Mobile App -> Home -> Accounts -> View Balance, or give a missed call to your registered balance-enquiry number for an instant SMS.'},
    {'id': 24, 'ticket_id': 'BK024', 'category': 'card', 'priority': 'high',
     'question': 'My ATM card got blocked after three wrong PIN attempts.',
     'response': 'Visit nearest branch with ID proof to unblock, or call 1800-419-1234. New PIN can also be generated via App -> Cards -> Generate PIN.'},
    {'id': 25, 'ticket_id': 'BK025', 'category': 'upi', 'priority': 'high',
     'question': 'Money got deducted from my account but the UPI payment shows failed.',
     'response': 'Check your bank SMS for the actual status. Failed UPI amounts are auto-refunded within T+2 days. Track via App -> UPI -> Transaction History.'},
]

# Append the new rows to our existing DataFrame
df_v2 = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

# Clean the questions in the same way as before (including the new rows)
df_v2['clean_question'] = df_v2['question'].apply(clean_text)

print('Dataset grew from', df.shape[0], 'rows to', df_v2.shape[0], 'rows.')
print()
print('Categories that now have 2 examples instead of 1:')
print(df_v2['category'].value_counts()[df_v2['category'].value_counts() > 1])

In [ ]:
# Re-fit the vectorizer on the EXTENDED dataset and train a new model on all 25 rows
vectorizer_v2 = TfidfVectorizer(max_features=500)
X_v2 = vectorizer_v2.fit_transform(df_v2['clean_question'])
y_v2 = df_v2['category']

model_v2 = MultinomialNB()
model_v2.fit(X_v2, y_v2)

print('New vocabulary size:', len(vectorizer_v2.get_feature_names_out()), 'words')
print('model_v2 trained on all', X_v2.shape[0], 'rows of the extended dataset.')

In [ ]:
# Fresh test questions - written in a THIRD style, never seen by either model
retrain_test_questions = {
    'account':  'I think someone locked my online banking account',
    'transfer': 'I want to do an NEFT transfer to my brother',
    'balance':  'Show me my savings balance',
    'card':     'My ATM card is blocked, what should I do?',
    'upi':      'money got deducted but payment shows failed',
}

before_correct = 0
after_correct = 0

print(f'{"True Category":15s} {"BEFORE (20 rows)":20s} {"AFTER (25 rows)":18s} Question')
print('-' * 100)

for true_category, question in retrain_test_questions.items():
    cleaned = clean_text(question)

    before_vec = vectorizer.transform([cleaned])
    before_pred = model_full.predict(before_vec)[0]

    after_vec = vectorizer_v2.transform([cleaned])
    after_pred = model_v2.predict(after_vec)[0]

    before_correct += (before_pred == true_category)
    after_correct += (after_pred == true_category)

    print(f'{true_category:15s} {before_pred:20s} {after_pred:18s} {question}')

print()
print(f'BEFORE (trained on 20 rows): {before_correct} / {len(retrain_test_questions)} correct')
print(f'AFTER  (trained on 25 rows): {after_correct} / {len(retrain_test_questions)} correct')

### Key Takeaway from Step 16

Adding just 5 well-chosen new examples improved performance on brand-new, never-before-seen questions. This is the ML feedback loop in action: more relevant training data directly improves real-world generalization, exactly the lesson Step 12 predicted. From this point forward, **`model_v2`, `vectorizer_v2`, and `df_v2` become our production chatbot brain** - this is the version we will save and deploy.

## STEP 17 - Save the Model

Training a model takes time and computing power. We do not want to retrain it every single time someone wants to use the chatbot. The solution is **model persistence**: save the trained model and vectorizer to disk as files, then simply load them back whenever needed, instantly, without retraining.

**Analogy:** think of a video game save file. You do not have to replay the entire game from the beginning every time you want to continue - you load your save file and pick up right where you left off. `joblib` lets us do the same thing with machine learning models.

In [ ]:
# Import joblib - the standard library for saving scikit-learn models
import joblib

# Save the trained model, the vectorizer, and the extended dataset to disk
joblib.dump(model_v2, 'banking_chatbot_model.pkl')
joblib.dump(vectorizer_v2, 'banking_chatbot_vectorizer.pkl')
df_v2.to_csv('banking_support_extended.csv', index=False)

print('Saved files:')
print('  banking_chatbot_model.pkl       - the trained Naive Bayes model')
print('  banking_chatbot_vectorizer.pkl  - the fitted TF-IDF vectorizer')
print('  banking_support_extended.csv    - the 25-row dataset used to look up responses')

In [ ]:
# Prove that loading works: load everything back into BRAND NEW variable names
# and confirm we can make a correct prediction WITHOUT retraining anything

loaded_model = joblib.load('banking_chatbot_model.pkl')
loaded_vectorizer = joblib.load('banking_chatbot_vectorizer.pkl')
loaded_df = pd.read_csv('banking_support_extended.csv')

test_response, test_category, test_confidence = get_response(
    'My UPI payment failed', loaded_model, loaded_vectorizer, loaded_df
)

print('Loaded model and vectorizer from disk successfully.')
print()
print('Category:', test_category, f'  Confidence: {test_confidence*100:.1f}%')
print('Response:', test_response)
print()
print('This worked instantly with zero training - exactly like loading a game save file.')

## STEP 18 - Build Chatbot UI with Gradio

So far our chatbot only works inside this notebook. **Gradio** lets us wrap any Python function in a simple, professional-looking web interface with just a few lines of code, and instantly generates a public shareable link so anyone (your classmates, your family, a recruiter) can try your chatbot from their own browser, on their own phone or laptop, without installing anything.

Run the cell below to install Gradio, then run the next cell to launch your chatbot. The link generated stays active for as long as this Colab notebook keeps running (typically up to 72 hours), and will stop working once you close the tab or the runtime disconnects.

In [ ]:
# Install Gradio (only needs to be run once per Colab session)
!pip install gradio -q

In [ ]:
import gradio as gr

def chatbot_response(user_question):
    """This is the function Gradio will call every time a user types a question."""
    response, category, confidence = get_response(
        user_question, loaded_model, loaded_vectorizer, loaded_df
    )
    return response

# Build the web interface
demo = gr.Interface(
    fn=chatbot_response,
    inputs=gr.Textbox(label='Ask your banking question', placeholder='e.g. My account is locked, what should I do?'),
    outputs=gr.Textbox(label='Chatbot Response'),
    title='Banking Support Chatbot',
    description='Ask any banking question (account, card, UPI, loan, KYC, and more) and get an instant answer.',
    examples=[
        'My account is locked, what should I do?',
        'How can I transfer money to another account?',
        'My debit card got declined at the shop.',
        'UPI transaction failed what to do?',
    ],
)

# Launch it! share=True generates a public link you can copy and share with anyone.
demo.launch(share=True, debug=False)

**Tip:** If you re-run the launch cell above and get a port-already-in-use type message, run `demo.close()` in a new cell first, then launch again. Copy your public `.gradio.live` link from the output above and share it with the class.

## STEP 19 - Stress Test the Chatbot

A model that works on a handful of hand-picked examples is not the same as a model that works reliably in the real world. Now we deliberately try to break it: a mix of clearly in-scope questions, oddly-phrased in-scope questions, and clearly out-of-scope questions.

**Classroom activity:** Open a classmate's shared Gradio link and try 5 questions of your own. Note down anything that gets misclassified or anything where the fallback message appears incorrectly (i.e. a real banking question that got rejected). Be ready to discuss: would more data fix it? Would better preprocessing fix it? Would a different/bigger model fix it?

In [ ]:
# A mix of easy, tricky, and out-of-scope questions to find weaknesses
stress_test_questions = [
    'My account is locked',                       # straightforward, in-scope
    'How do I open an FD',                        # short, in-scope
    'What documents do I need for KYC',            # paraphrased, in-scope
    'My UPI transaction failed',                   # straightforward, in-scope
    'What is the weather in Delhi today',          # clearly out-of-scope
    'Can you sing a song for me',                  # clearly out-of-scope
    'I want to close my account permanently',      # tricky - not in our 20 categories' exact wording
]

for q in stress_test_questions:
    response, category, confidence = get_response(q, loaded_model, loaded_vectorizer, loaded_df)
    conf_display = f'{confidence*100:.1f}%' if category is not None else 'n/a'
    print(f'Question: {q}')
    print(f'  -> Predicted category: {category}   Confidence: {conf_display}')
    print(f'  -> Response: {response[:90]}{"..." if len(response) > 90 else ""}')
    print()

### What Did We Find?

Run the cell above and look closely - you will likely see at least one in-scope question get classified into the wrong category (for example, a question about a category we still only have 1-2 examples for may get confused with a similar-sounding one). This is expected and is the entire point of stress testing.

**Group discussion:**
- **More data** would directly help: every misclassified category here still only has 1 or 2 training examples. Real banking chatbots train on thousands of real customer messages per category.
- **Better preprocessing** could help too: techniques like stemming or lemmatization (reducing words like "transferring" and "transferred" to "transfer") would let the model recognize more word variations than our simple lowercase-and-stopword-removal approach.
- **A bigger/different model** is the least impactful fix here. As Step 15 showed, switching algorithms cannot compensate for a genuine shortage of training examples.

## STEP 20 - What's Next - Course Roadmap

Congratulations - you have built a complete, working, deployed Machine Learning product from scratch: raw CSV data, all the way to a live shareable chatbot with a public link. That is a real accomplishment, and a genuine portfolio piece you can show your parents, post on LinkedIn, and talk about in interviews.

**But this is just the beginning of the iceberg.** Here is how today's work connects to the much bigger world of AI:

| Layer | What It Covers | How Today Connects |
|-------|----------------|---------------------|
| **Python Fundamentals** | Variables, loops, functions, data structures | You already used all of this today inside Pandas and our custom functions |
| **Mathematics for ML** | Probability, linear algebra, calculus, statistics | Naive Bayes used probability theory; TF-IDF used vector math; deeper models need calculus for training |
| **Classical Machine Learning** | Regression, classification, clustering, ensembles, model tuning | You did classification today - this is one of dozens of techniques in this layer |
| **Deep Learning** | Neural networks, CNNs, RNNs, backpropagation | The natural next step once classical ML hits its limits, especially on large unstructured data |
| **Transformers** | Attention mechanism, BERT, GPT-style architectures | The technology actually powering modern AI chatbots like ChatGPT and Claude - far more powerful than TF-IDF + Naive Bayes |
| **RAG (Retrieval-Augmented Generation)** | Combining a knowledge base with a language model | The modern, more powerful version of exactly what we built today: predict intent and retrieve the right information |
| **AI Agents** | Models that can use tools, take multi-step actions, and make decisions | Where the entire industry is heading next - chatbots that do not just answer, but act |

**Today, in one session, you learned a working slice of Steps from across this entire roadmap.** A full course goes ten times deeper into every single layer above - more data, more algorithms, the math behind why they work, and the modern architectures running in production at real companies today.

---

## Workshop Complete - All 20 Steps Done!

| Step | What We Did | Result |
|------|-------------|--------|
| 1-2 | Big picture + Colab setup | Understood the full roadmap |
| 3-4 | Loaded and inspected the data | DataFrame with 20 rows, 6 columns |
| 5-6 | EDA - category, priority, question length | Visualized the data |
| 7 | Text preprocessing | Clean, lowercase, stopword-free text |
| 8 | TF-IDF vectorization | Feature matrix X and labels y |
| 9 | Train-test split | Honest, unseen evaluation data |
| 10 | Trained Naive Bayes | A working classifier |
| 11 | Made predictions | Live, real-time classification |
| 12 | Evaluated the model | Accuracy, precision, recall, confusion matrix, and a key lesson about data quantity |
| 13-14 | Response retrieval + fallback handling | The complete core chatbot function |
| 15 | Compared 3 different models | Confirmed no model fixes a data shortage |
| 16 | Added more data and retrained | Measurable accuracy improvement on new questions |
| 17 | Saved the model with joblib | Instant load, no retraining needed |
| 18 | Built a Gradio web app | A live, public, shareable chatbot link |
| 19 | Stress tested the chatbot | Found real weaknesses, discussed real fixes |
| 20 | Course roadmap | Connected today's work to the full AI landscape |

### Before you leave today

- Save this notebook now: **File > Save a Copy in Drive**
- Rename it to **banking_chatbot_yourname.ipynb**
- Copy your Gradio public link somewhere safe (it expires when this Colab session ends)
- Write three lines in a text cell: what you did today, what you understood, what confused you

You built a real, deployed Machine Learning product today. Well done.

---